In [1]:
import pandas as pd

### Key points: groupby basics (Block 1)

- `groupby("city")` splits the rows into one pile for each distinct city. Nothing is calculated until you pick a column and a method.
- The pattern is: `df.groupby("group column")["number column"].method()`.
- Common methods: `sum()`, `mean()`, `max()`, `min()`, `count()` (non-empty cells), and `size()` (rows in each pile).
- The result is a Series. The group names (Delhi, Indore, Pune) become its labels, so everything from Topic 7 still works on it: `sort_values`, `idxmax()`, `["Pune"]`.
- `.reset_index()` or `groupby("city", as_index=False)` turns the group names back into a normal column.
- `groupby(...).mean()` without choosing a column tries every column and fails on text columns. Name the column.
- Groups are sorted by the group name by default.
- A wrong group column name gives KeyError.
- Group totals add up to the grand total, which is a quick sanity check: `orders["amount"].sum()`.
- `groupby` never changes the original table.

In [12]:
orders = pd.DataFrame({"order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],       # 8 bills
                       "customer": ["asha", "ravi", "zoya", "meena", "asha", "zoya", "ravi", "kiran"],
                       "city": ["Indore", "Delhi", "Pune", "Indore", "Indore", "Pune", "Delhi", "Pune"],
                       "category": ["Food", "Toys", "Books", "Food", "Books", "Food", "Toys", "Toys"],
                       "amount": [1200, 800, 1500, 950, 400, 300, 600, 1100]})


print(orders.groupby("city")["amount"].sum())
print(orders.groupby("city")["amount"].mean())
print(orders.groupby("city")["amount"].max())
print(orders.groupby("city").size())

top = orders.groupby("city")["amount"].sum().sort_values(ascending=False)
print(top)
print(top.idxmax())  
print(top["Pune"])


print(orders.groupby("city")["amount"].sum().reset_index())   

city
Delhi     1400
Indore    2550
Pune      2900
Name: amount, dtype: int64
city
Delhi     700.000000
Indore    850.000000
Pune      966.666667
Name: amount, dtype: float64
city
Delhi      800
Indore    1200
Pune      1500
Name: amount, dtype: int64
city
Delhi     2
Indore    3
Pune      3
dtype: int64
city
Pune      2900
Indore    2550
Delhi     1400
Name: amount, dtype: int64
Pune
2900
     city  amount
0   Delhi    1400
1  Indore    2550
2    Pune    2900


### groupby + agg
- `agg` = fill one form (many summaries) for every group, in one table.
- Best way: `df.groupby("g").agg(new_name=("column", "method"))`
- Methods: sum, mean, max, min, count, size, nunique (nunique = different values counted once).
- Result is a normal table with group names as labels:
  `.reset_index()`, `.sort_values("new_name")`, `.loc[group, "new_name"]`, `.idxmax()` work.
- Older ways: list `["sum","mean"]` and dict `{"col":"method"}`.
- Watch out: wrong column name -> KeyError. Missing ("column","method") -> TypeError.

In [9]:
# the table (8 bills)
orders = pd.DataFrame({"order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
                       "customer": ["asha", "ravi", "zoya", "meena", "asha", "zoya", "ravi", "kiran"],
                       "city": ["Indore", "Delhi", "Pune", "Indore", "Indore", "Pune", "Delhi", "Pune"],
                       "category": ["Food", "Toys", "Books", "Food", "Books", "Food", "Toys", "Toys"],
                       "amount": [1200, 800, 1500, 950, 400, 300, 600, 1100]})

r = orders.groupby("city").agg(
    total = ("amount", "sum"),
    biggest = ("amount","max"),
    bills = ("order_id","count"),
    buyers = ("customer","nunique"),
    )

print(r)


print(r.reset_index())
print(r.sort_values("total",ascending=False))
print(r.loc["Pune","total"])
print(r["total"].idxmax())

print(orders.groupby("city")["amount"].agg(["sum", "mean", "max"]))
print(orders.groupby("city")["amount"].agg(["sum", "mean"]).round(1))


        total  biggest  bills  buyers
city                                 
Delhi    1400      800      2       1
Indore   2550     1200      3       2
Pune     2900     1500      3       2
     city  total  biggest  bills  buyers
0   Delhi   1400      800      2       1
1  Indore   2550     1200      3       2
2    Pune   2900     1500      3       2
        total  biggest  bills  buyers
city                                 
Pune     2900     1500      3       2
Indore   2550     1200      3       2
Delhi    1400      800      2       1
2900
Pune
         sum        mean   max
city                          
Delhi   1400  700.000000   800
Indore  2550  850.000000  1200
Pune    2900  966.666667  1500
         sum   mean
city               
Delhi   1400  700.0
Indore  2550  850.0
Pune    2900  966.7


### groupby: two columns, filter groups, top rows
- Two group columns go in ONE list: `df.groupby(["a","b"])["x"].sum()`; result has two labels -> `.reset_index()` or `as_index=False`.
- `agg` works with two group columns too.
- Keep only some groups, original rows: `df.groupby("g").filter(lambda g: g["x"].sum() > 2000)`.
- Same result in 2 steps: `t = df.groupby("g")["x"].sum()`, `big = t[t > 2000].index`, `df[df["g"].isin(big)]`.
- Top n rows per group: sort first, then `df.sort_values("x", ascending=False).groupby("g").head(n)`.
- Biggest row per group: `df.loc[df.groupby("g")["x"].idxmax()]`.
- Watch out: `groupby("a","b")` is wrong -> use a list. Without sorting first, `head(n)` is not "top n".

In [14]:
print(orders.groupby(["city","category"])["amount"].sum())

print(orders.groupby(["city","category"])["amount"].sum().reset_index())

print(orders.groupby(["city", "category"], as_index=False).agg(
    total=("amount", "sum"),
    bills=("order_id", "count"),
))

city    category
Delhi   Toys        1400
Indore  Books        400
        Food        2150
Pune    Books       1500
        Food         300
        Toys        1100
Name: amount, dtype: int64
     city category  amount
0   Delhi     Toys    1400
1  Indore    Books     400
2  Indore     Food    2150
3    Pune    Books    1500
4    Pune     Food     300
5    Pune     Toys    1100
     city category  total  bills
0   Delhi     Toys   1400      2
1  Indore    Books    400      1
2  Indore     Food   2150      2
3    Pune    Books   1500      1
4    Pune     Food    300      1
5    Pune     Toys   1100      1


In [16]:
orders.groupby("city").filter(lambda g: g["amount"].sum() >2000 )

,order_id,customer,city,category,amount
0,1001,asha,Indore,Food,1200
2,1003,zoya,Pune,Books,1500
3,1004,meena,Indore,Food,950
4,1005,asha,Indore,Books,400
5,1006,zoya,Pune,Food,300
7,1008,kiran,Pune,Toys,1100


In [17]:
s = orders.sort_values("amount", ascending=False)   # biggest bills first
s.groupby("city").head(1)                           # first 1 row of each city

,order_id,customer,city,category,amount
2,1003,zoya,Pune,Books,1500
0,1001,asha,Indore,Food,1200
1,1002,ravi,Delhi,Toys,800


In [18]:
s.groupby("city").head(2)

,order_id,customer,city,category,amount
2,1003,zoya,Pune,Books,1500
0,1001,asha,Indore,Food,1200
7,1008,kiran,Pune,Toys,1100
3,1004,meena,Indore,Food,950
1,1002,ravi,Delhi,Toys,800
6,1007,ravi,Delhi,Toys,600


In [19]:
orders.loc[orders.groupby("city")["amount"].idxmax()]

,order_id,customer,city,category,amount
1,1002,ravi,Delhi,Toys,800
0,1001,asha,Indore,Food,1200
2,1003,zoya,Pune,Books,1500


### groupby + transform
- `transform` = the group's number written back on EVERY original row (same length as the table).
- `df["new"] = df.groupby("g")["x"].transform("sum")` (or "mean", "max", "min", "count").
- Sum/agg give one row per group; transform gives one value per row, in the original row order.
- Use it to compare each row with its own group: `df["x"] > df["group_avg"]`.
- Common in ML to make features like "average price of this city".
- Watch out: using `.sum()` instead of `.transform("sum")` to make a column gives all NaN (no error).
- Wrong method name -> ValueError.

In [20]:
orders.groupby("city")["amount"].sum()

city
Delhi     1400
Indore    2550
Pune      2900
Name: amount, dtype: int64

In [21]:
orders.groupby("city")["amount"].transform("sum")

0    2550
1    1400
2    2900
3    2550
4    2550
5    2900
6    1400
7    2900
Name: amount, dtype: int64

In [22]:
orders.groupby("city")["amount"].transform("sum").reset_index()

,index,amount
0,0,2550
1,1,1400
2,2,2900
3,3,2550
4,4,2550
5,5,2900
6,6,1400
7,7,2900


In [24]:
o = orders.copy()
o["city_total"] = o.groupby("city")["amount"].transform("sum")
o["city_avg"]   = o.groupby("city")["amount"].transform("mean").round(1)
print(o)

   order_id customer    city category  amount  city_total  city_avg
0      1001     asha  Indore     Food    1200        2550     850.0
1      1002     ravi   Delhi     Toys     800        1400     700.0
2      1003     zoya    Pune    Books    1500        2900     966.7
3      1004    meena  Indore     Food     950        2550     850.0
4      1005     asha  Indore    Books     400        2550     850.0
5      1006     zoya    Pune     Food     300        2900     966.7
6      1007     ravi   Delhi     Toys     600        1400     700.0
7      1008    kiran    Pune     Toys    1100        2900     966.7


In [27]:
o["above_avg"] = o["amount"] > o["city_avg"]
print(o)
o[o["amount"] > o["city_avg"]] 

   order_id customer    city category  amount  city_total  city_avg  above_avg
0      1001     asha  Indore     Food    1200        2550     850.0       True
1      1002     ravi   Delhi     Toys     800        1400     700.0       True
2      1003     zoya    Pune    Books    1500        2900     966.7       True
3      1004    meena  Indore     Food     950        2550     850.0       True
4      1005     asha  Indore    Books     400        2550     850.0      False
5      1006     zoya    Pune     Food     300        2900     966.7      False
6      1007     ravi   Delhi     Toys     600        1400     700.0      False
7      1008    kiran    Pune     Toys    1100        2900     966.7       True


,order_id,customer,city,category,amount,city_total,city_avg,above_avg
0,1001,asha,Indore,Food,1200,2550,850.0,True
1,1002,ravi,Delhi,Toys,800,1400,700.0,True
2,1003,zoya,Pune,Books,1500,2900,966.7,True
3,1004,meena,Indore,Food,950,2550,850.0,True
7,1008,kiran,Pune,Toys,1100,2900,966.7,True


### groupby + transform: share, rank, fill
- Share of group total: `df["total"] = df.groupby("g")["x"].transform("sum")`, then `df["share"] = df["x"] / df["total"] * 100`.
- Rank inside each group: `df.groupby("g")["x"].rank(ascending=False).astype(int)`; rank 1 = biggest, restarts in every group.
- Top n per group using the rank column: `df[df["rank"] <= n]`.
- Fill gaps with the group average: `df["x"].fillna(df.groupby("g")["x"].transform("mean"))`.
- NaN is skipped when computing the group mean.
- Watch out: forgetting `groupby` gives a whole-table rank/mean (no error). Using `df["x"].mean()` to fill ignores the group.

In [31]:
o = orders.copy()
o["city_total"] = o.groupby("city")["amount"].transform("sum")
print(o)
o = orders.copy()
o["city_total"] = o.groupby("city")["amount"].transform("sum")   # Block 4
o["share_pct"]  = (o["amount"] / o["city_total"] * 100).round(1)  # slice / pizza * 100
print(o[["order_id", "city", "amount", "city_total", "share_pct"]])

   order_id customer    city category  amount  city_total
0      1001     asha  Indore     Food    1200        2550
1      1002     ravi   Delhi     Toys     800        1400
2      1003     zoya    Pune    Books    1500        2900
3      1004    meena  Indore     Food     950        2550
4      1005     asha  Indore    Books     400        2550
5      1006     zoya    Pune     Food     300        2900
6      1007     ravi   Delhi     Toys     600        1400
7      1008    kiran    Pune     Toys    1100        2900
   order_id    city  amount  city_total  share_pct
0      1001  Indore    1200        2550       47.1
1      1002   Delhi     800        1400       57.1
2      1003    Pune    1500        2900       51.7
3      1004  Indore     950        2550       37.3
4      1005  Indore     400        2550       15.7
5      1006    Pune     300        2900       10.3
6      1007   Delhi     600        1400       42.9
7      1008    Pune    1100        2900       37.9


In [33]:
o["rank_in_city"] = o.groupby("city")["amount"].rank(ascending=False).astype(int)
print(o)

   order_id customer    city category  amount  city_total  share_pct  \
0      1001     asha  Indore     Food    1200        2550       47.1   
1      1002     ravi   Delhi     Toys     800        1400       57.1   
2      1003     zoya    Pune    Books    1500        2900       51.7   
3      1004    meena  Indore     Food     950        2550       37.3   
4      1005     asha  Indore    Books     400        2550       15.7   
5      1006     zoya    Pune     Food     300        2900       10.3   
6      1007     ravi   Delhi     Toys     600        1400       42.9   
7      1008    kiran    Pune     Toys    1100        2900       37.9   

   rank_in_city  
0             1  
1             1  
2             1  
3             2  
4             3  
5             3  
6             2  
7             2  


In [34]:
o[o["rank_in_city"] == 1][["order_id", "city", "amount"]]


,order_id,city,amount
0,1001,Indore,1200
1,1002,Delhi,800
2,1003,Pune,1500


In [35]:
import numpy as np
m = orders.copy()
m["amount"] = m["amount"].astype("float")   # float, so it can hold NaN
m.loc[[1, 5], "amount"] = np.nan            # Delhi bill and Pune bill are now empty